# 03. Image Resize & Aspect Ratio Analysis

Mục tiêu:

1. Kiểm tra kích thước và aspect ratio của ảnh SOP.
2. Phát hiện ảnh có aspect ratio bất thường.
3. Resize ảnh nhưng giữ nguyên aspect ratio.
4. So sánh:
   - distortion của direct resize
   - distortion của aspect-ratio-preserving resize.
5. Lưu metadata preprocessing.

Không resize trực tiếp từ `(W,H)` về `(224,224)` vì có thể làm biến dạng sản phẩm.

## 1. Configuration

In [ ]:
!pip install -q pandas numpy pillow opencv-python tqdm matplotlib

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import cv2

from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


PROJECT_ROOT = Path.cwd()

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Stanford_Online_Products"
)

SAMPLE_DIR = (
    PROJECT_ROOT
    / "data"
    / "sampled"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "03_resize"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_SIZE = 224

TRAIN_CSV = SAMPLE_DIR / "train_10k.csv"
TEST_CSV = SAMPLE_DIR / "test_10k.csv"

## Load metadata

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

df = pd.concat(
    [
        train_df.assign(split="train"),
        test_df.assign(split="test")
    ],
    ignore_index=True
)

print(df.shape)

## Image statistics

In [ ]:
def get_image_info(path):
    image = Image.open(path)
    width, height = image.size
    return {
        "width": width,
        "height": height,
        "aspect_ratio": width / height
    }

records = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    path = DATASET_ROOT / row["path"]
    try:
        info = get_image_info(path)
        records.append({
            "image_id": row["image_id"],
            **info
        })
    except Exception as e:
        records.append({
            "image_id": row["image_id"],
            "width": None,
            "height": None,
            "aspect_ratio": None
        })

image_stats = pd.DataFrame(records)
image_stats.describe()

## Aspect ratio disstribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(
    image_stats["aspect_ratio"].dropna(),
    bins=50
)
plt.xlabel("Aspect Ratio (W/H)")
plt.ylabel("Number of Images")
plt.title("SOP Aspect Ratio Distribution")

plt.show()

## Aspect-ratio-preserving resize

In [ ]:
def resize_keep_aspect(image,target_size=224):
    h, w = image.shape[:2]
    scale = min(
        target_size / w,
        target_size / h
    )
    new_w = round(w * scale)
    new_h = round(h * scale)
    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )
    return resized

## Letter box

In [ ]:
def letterbox(image,target_size=224,pad_value=0):
    h, w = image.shape[:2]
    scale = min(
        target_size / w,
        target_size / h
    )
    new_w = round(w * scale)
    new_h = round(h * scale)
    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )
    canvas = np.full(
        (
            target_size,
            target_size,
            3
        ),
        pad_value,
        dtype=np.uint8
    )
    x = (target_size - new_w) // 2
    y = (target_size - new_h) // 2
    canvas[
        y:y + new_h,
        x:x + new_w
    ] = resized
    return canvas

## Visualization

In [ ]:
sample = df.sample(5, random_state=42)
fig, axes = plt.subplots(5,2,figsize=(8, 15))

for i, (_, row) in enumerate(sample.iterrows()):
    path = DATASET_ROOT / row["path"]
    image = cv2.imread(str(path))
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    processed = letterbox(
        image,
        TARGET_SIZE
    )

    axes[i, 0].imshow(image)
    axes[i, 0].axis("off")
    axes[i, 0].set_title("Original")

    axes[i, 1].imshow(processed)
    axes[i, 1].axis("off")
    axes[i, 1].set_title("Letterbox")

plt.tight_layout()

## Save

In [ ]:
OUTPUT_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(exist_ok=True)

metadata = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    src = DATASET_ROOT / row["path"]
    image = cv2.imread(str(src))
    if image is None:
        continue
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
    processed = letterbox(
        image,
        TARGET_SIZE
    )
    out_name = (
        f'{row["image_id"]}.jpg'
    )
    out_path = OUTPUT_DIR / out_name
    Image.fromarray(processed).save(
        out_path,
        quality=95
    )
    metadata.append({
        "image_id": row["image_id"],
        "class_id": row["class_id"],
        "split": row["split"],
        "original_path": row["path"],
        "processed_path": str(out_path)
    })

pd.DataFrame(metadata).to_csv(
    OUTPUT_DIR.parent / "metadata.csv",
    index=False
)